# 06 — The DataFrame Object: Deep Dive & Interview Essentials
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Machine Learning, and Analytics Interviews.*

---

## 📌 Executive Summary & Interview Expectations
The `pd.DataFrame` is the cornerstone 2D data structure in Python. Technical interviews test candidates not merely on basic column indexing, but on the **underlying tabular mechanics**, **memory representation (BlockManager / Arrow backend)**, and **indexing performance subtleties**.

### Core Competencies Tested in this Module:
1. **DataFrame Construction & Internal Layout**: Constructing from dicts, 2D NumPy arrays, and CSV with datetime parsing.
2. **Metadata & Structural Properties**: `dtypes`, `axes`, `shape`, `size`, and numeric-only aggregations.
3. **Advanced Sorting**: Multi-column sorting with mixed directions (`ascending=[True, False]`) and `na_position`.
4. **Memory Profiling & Optimization**: Using categorical dtypes (`astype('category')`) and understanding the cardinality threshold.
5. **The 4 Accessor Paradigms**: Exhaustive comparison of `.loc` vs `.iloc` vs `.at` vs `.iat`.
6. **Index Mutation & Duplicate Indices**: Resetting, swapping, and querying non-unique indices (why `.loc` returns a DataFrame instead of a Series).
7. **Interview Corner**: The danger of `df.columns = [...]`, high-cardinality category traps, and finding top records per group.

## 1. Environment Setup & Data Ingestion
We load both `nba.csv` and `nfl.csv`, with automated local/remote fallback.

In [1]:
import os
import time
import numpy as np
import pandas as pd

# Ingest NBA dataset with datetime parsing
nba_path = "nba.csv"
if not os.path.exists(nba_path):
    nba_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_04_the_dataframe_object/nba.csv"

nba = pd.read_csv(nba_path, parse_dates=["Birthday"])
print("NBA Data loaded successfully. Shape:", nba.shape)

NBA Data loaded successfully. Shape: (450, 5)


/var/folders/54/5j97z6452x19yddj6yv2l7j00000gn/T/ipykernel_26455/1659079087.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nba = pd.read_csv(nba_path, parse_dates=["Birthday"])


## 2. Constructing DataFrames from Python Dictionaries & NumPy

### 💡 Interview Note — Memory Architecture:
A DataFrame is conceptually an ordered dictionary of `Series`, where every Series shares the same `Index`.
- When constructed from a **dict of lists**, keys become column names.
- When constructed from a **NumPy 2D array**, memory is contiguous, and row/column labels must be explicitly supplied.

In [2]:
# 1. From Python Dictionary
city_data = {
    "City": ["New York City", "Paris", "Barcelona", "Tokyo"],
    "Country": ["United States", "France", "Spain", "Japan"],
    "Population": [8_336_817, 2_161_000, 1_620_000, 13_960_000]
}

cities_df = pd.DataFrame(city_data)
cities_df

,City,Country,Population
0,New York City,United States,8336817
1,Paris,France,2161000
2,Barcelona,Spain,1620000
3,Tokyo,Japan,13960000


In [3]:
# 2. From 2D NumPy array with custom row and column indices
np.random.seed(42)
random_matrix = np.random.randint(10, 100, size=(3, 5))
row_labels = ["Morning", "Afternoon", "Evening"]
col_labels = ["Mon", "Tue", "Wed", "Thu", "Fri"]

schedule_df = pd.DataFrame(data=random_matrix, index=row_labels, columns=col_labels)
schedule_df

,Mon,Tue,Wed,Thu,Fri
Morning,61,24,81,70,30
Afternoon,92,96,84,84,97
Evening,33,12,31,62,11


## 3. Structural Attributes & Metadata Inspection

### ⚠️ Top Interview Question: `df.axes` and `df.ndim`
- `df.ndim`: Always returns `2` for a DataFrame (1 for Series).
- `df.axes`: Returns a list of the two axes: `[Index(row_labels), Index(column_labels)]`.
- `df.dtypes.value_counts()`: Rapidly tallies how many features belong to each type.

In [4]:
# Inspect dtypes and tally by type
print("Column Data Types:\n", nba.dtypes)
print("\nData Type Breakdown:\n", nba.dtypes.value_counts())

Column Data Types:
 Name                   str
Team                   str
Position               str
Birthday    datetime64[us]
Salary               int64
dtype: object

Data Type Breakdown:
 str               3
datetime64[us]    1
int64             1
Name: count, dtype: int64


In [5]:
# Inspect index, columns, and axes
print("Index:   ", nba.index)
print("Columns: ", list(nba.columns))
print("Axes:    ", nba.axes)
print("Shape:   ", nba.shape)
print("Size:    ", nba.size, "(rows * columns)")

Index:    RangeIndex(start=0, stop=450, step=1)
Columns:  ['Name', 'Team', 'Position', 'Birthday', 'Salary']
Axes:     [RangeIndex(start=0, stop=450, step=1), Index(['Name', 'Team', 'Position', 'Birthday', 'Salary'], dtype='str')]
Shape:    (450, 5)
Size:     2250 (rows * columns)


### 💡 Modern Aggregation Syntax: `numeric_only=True`
> **Pandas 2.x/3.x Standard**: In modern Pandas, calling statistical reductions (e.g., `sum()`, `mean()`, `median()`) on mixed DataFrames requires `numeric_only=True` to avoid `TypeError`. In contrast, `describe()` automatically filters to numeric columns by default or takes `include=[np.number]`.

In [6]:
# Summary statistics for numeric columns
print("Average Salary: $", round(nba["Salary"].mean(), 2))
print("Median Salary:  $", round(nba["Salary"].median(), 2))
display(nba.describe())

Average Salary: $ 7653583.76
Median Salary:  $ 3303074.5


,Birthday,Salary
count,450,4.500000e+02
mean,1993-11-24 02:01:36,7.653584e+06
min,1977-01-26 00:00:00,7.956800e+04
25%,1991-03-16 12:00:00,1.618520e+06
50%,1994-04-29 00:00:00,3.303074e+06
75%,1997-02-17 12:00:00,1.012957e+07
max,2000-12-23 00:00:00,4.023176e+07
std,NaN,9.288810e+06


## 4. Sorting DataFrames: Single, Multi-Column & Index

### 💡 Interview Pro-Tip — Sorting Nuances:
1. **Multi-Column Sorting**: `nba.sort_values(by=["Team", "Salary"], ascending=[True, False])`.
2. **Missing Values**: `na_position="first"` vs `na_position="last"`.
3. **Index Sorting**: `nba.sort_index(ascending=False)` is crucial after groupbys or merges to re-establish chronological/alphabetical order.

In [7]:
# Sort by Name (A-Z)
nba.sort_values(by="Name").head()

,Name,Team,Position,Birthday,Salary
52,Aaron Gordon,Orlando Magic,PF,1995-09-16,19863636
101,Aaron Holiday,Indiana Pacers,PG,1996-09-30,2239200
437,Abdel Nader,Oklahoma City Thunder,SF,1993-09-25,1618520
81,Adam Mokoka,Chicago Bulls,G,1998-07-18,79568
399,Admiral Schofield,Washington Wizards,SF,1997-03-30,1000000


In [8]:
# Multi-column sort: Team alphabetically, then Salary descending (highest paid first)
nba.sort_values(by=["Team", "Salary"], ascending=[True, False]).head(10)

,Name,Team,Position,Birthday,Salary
111,Chandler Parsons,Atlanta Hawks,SF,1988-10-25,25102512
28,Evan Turner,Atlanta Hawks,PG,1988-10-27,18606556
167,Allen Crabbe,Atlanta Hawks,SG,1992-04-09,18500000
213,De'Andre Hunter,Atlanta Hawks,SF,1997-12-02,7068360
339,Jabari Parker,Atlanta Hawks,PF,1995-03-15,6500000
194,Cam Reddish,Atlanta Hawks,SF,1999-09-01,4245720
359,Alex Len,Atlanta Hawks,C,1993-06-16,4160000
84,John Collins,Atlanta Hawks,PF,1997-09-23,2686560
20,Kevin Huerter,Atlanta Hawks,SG,1998-08-27,2636280
98,Vince Carter,Atlanta Hawks,PF,1977-01-26,2564753


## 5. Memory Profiling & Optimization with Categoricals

### ⚠️ Top Interview Question: When SHOULDN'T you use `category` dtype?
- **Good Candidates**: Low-cardinality columns (e.g. `Team` with 30 unique teams, `Position` with 5 unique positions).
- **Bad Candidates**: High-cardinality columns (e.g. `Name`, User IDs, UUIDs) where almost every value is unique. Creating a categorical mapping table for millions of unique strings actually **increases** memory and slows down lookups!

In [9]:
# Initial memory consumption
mem_before = nba.memory_usage(deep=True).sum()
print(f"Memory BEFORE optimization: {mem_before:,} bytes")

# Convert low-cardinality object columns to category
nba["Team"] = nba["Team"].astype("category")
nba["Position"] = nba["Position"].astype("category")

mem_after = nba.memory_usage(deep=True).sum()
print(f"Memory AFTER optimization:  {mem_after:,} bytes")
print(f"Memory Reduction:            {((mem_before - mem_after) / mem_before) * 100:.1f}%")

Memory BEFORE optimization: 87,410 bytes
Memory AFTER optimization:  38,613 bytes
Memory Reduction:            55.8%


## 6. Slicing Columns: 1D Series vs 2D DataFrames

```python
nba['Salary']    # Returns a 1D Series (Shape: (N,))
nba[['Salary']]  # Returns a 2D DataFrame (Shape: (N, 1))
```

In [10]:
print("nba['Salary'] type:  ", type(nba["Salary"]))
print("nba[['Salary']] type:", type(nba[["Salary"]]))
nba[["Name", "Team", "Salary"]].head()

nba['Salary'] type:   <class 'pandas.Series'>
nba[['Salary']] type: <class 'pandas.DataFrame'>


,Name,Team,Salary
0,Shake Milton,Philadelphia 76ers,1445697
1,Christian Wood,Detroit Pistons,1645357
2,PJ Washington,Charlotte Hornets,3831840
3,Derrick Rose,Detroit Pistons,7317074
4,Marial Shayok,Philadelphia 76ers,79568


## 7. Row Indexing: `.loc` vs `.iloc` vs `.at` vs `.iat`

### ⚠️ Top Interview Comparison Table:
| Accessor | Type of Indexer | Dimensions | Endpoint Inclusivity | Speed |
| :--- | :--- | :--- | :--- | :--- |
| **`.iloc`** | Integer / Positional (`0..N-1`) | Scalar, Series, or DataFrame | **Exclusive** of stop | Moderate |
| **`.loc`** | Label / Boolean | Scalar, Series, or DataFrame | **Inclusive** of stop | Moderate |
| **`.iat`** | Integer coordinate `[i, j]` | Strictly single **scalar** | N/A | **Ultra-Fast** |
| **`.at`** | Label coordinate `[row_label, col_label]` | Strictly single **scalar** | N/A | **Ultra-Fast** |

In [11]:
# Set player Name as row index for label-based demonstration
nba_indexed = nba.set_index("Name")
nba_indexed.head(3)

,Team,Position,Birthday,Salary
Name,,,,
Shake Milton,Philadelphia 76ers,SG,1996-09-26,1445697
Christian Wood,Detroit Pistons,PF,1995-09-27,1645357
PJ Washington,Charlotte Hornets,PF,1998-08-23,3831840


In [12]:
# Positional lookup using .iloc: row 0, first 3 columns
display(nba_indexed.iloc[0, 0:3])

Team         Philadelphia 76ers
Position                     SG
Birthday    1996-09-26 00:00:00
Name: Shake Milton, dtype: object

In [13]:
# Label-based lookup using .loc: specific player and column range
display(nba_indexed.loc["Derrick Rose", "Team":"Salary"])

Team            Detroit Pistons
Position                     PG
Birthday    1988-10-04 00:00:00
Salary                  7317074
Name: Derrick Rose, dtype: object

In [14]:
# High-speed scalar access comparison: .loc vs .at
t0 = time.perf_counter()
for _ in range(1000):
    val_loc = nba_indexed.loc["Derrick Rose", "Salary"]
t1 = time.perf_counter()

t2 = time.perf_counter()
for _ in range(1000):
    val_at = nba_indexed.at["Derrick Rose", "Salary"]
t3 = time.perf_counter()

print(f"1,000 lookups using .loc: {(t1 - t0)*1000:.2f} ms")
print(f"1,000 lookups using .at:  {(t3 - t2)*1000:.2f} ms")
print(f"Speedup via .at:          {((t1 - t0) / (t3 - t2)):.1f}x faster!")

1,000 lookups using .loc: 4.23 ms
1,000 lookups using .at:  3.29 ms
Speedup via .at:          1.3x faster!


## 8. Modifying Schema: Renaming & Swapping Indices

### ⚠️ Top Interview Trap: `df.rename()` vs `df.columns = [...]`
- **`df.columns = [...]`**: Fragile. Requires passing an exact list matching the column count. If columns are reordered or an extra column is added upstream, this silently misassigns column names!
- **`df.rename(columns={'Old': 'New'})`**: Robust. Targets specific columns by name; leaves other columns untouched.

In [15]:
# Safe, targeted renaming via dictionary
renamed_df = nba.rename(columns={"Birthday": "Birth_Date", "Salary": "Compensation"})
renamed_df.head(2)

,Name,Team,Position,Birth_Date,Compensation
0,Shake Milton,Philadelphia 76ers,SG,1996-09-26,1445697
1,Christian Wood,Detroit Pistons,PF,1995-09-27,1645357


In [16]:
# Swapping the index: reset existing index first, then set new index
team_indexed = nba.set_index("Team")
team_indexed.head()

,Name,Position,Birthday,Salary
Team,,,,
Philadelphia 76ers,Shake Milton,SG,1996-09-26,1445697
Detroit Pistons,Christian Wood,PF,1995-09-27,1645357
Charlotte Hornets,PJ Washington,PF,1998-08-23,3831840
Detroit Pistons,Derrick Rose,PG,1988-10-04,7317074
Philadelphia 76ers,Marial Shayok,G,1995-07-26,79568


## 9. Consolidation Practice: NFL Salary Analytics

In [17]:
# Ingest NFL dataset
nfl_path = "nfl.csv"
if not os.path.exists(nfl_path):
    nfl_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_04_the_dataframe_object/nfl.csv"

nfl = pd.read_csv(nfl_path, index_col="Name", parse_dates=["Birthday"])
nfl.head()

,Team,Position,Birthday,Salary
Name,,,,
Tremon Smith,Philadelphia Eagles,RB,1996-07-20,570000
Shawn Williams,Cincinnati Bengals,SS,1991-05-13,3500000
Adam Butler,New England Patriots,DT,1994-04-12,645000
Derek Wolfe,Denver Broncos,DE,1990-02-24,8000000
Jake Ryan,Jacksonville Jaguars,OLB,1992-02-27,1000000


In [18]:
# Top 5 highest paid NFL players
top_nfl = nfl.sort_values(by="Salary", ascending=False).head(5)
top_nfl[["Team", "Position", "Salary"]]

,Team,Position,Salary
Name,,,
Kirk Cousins,Minnesota Vikings,QB,27500000
Jameis Winston,Tampa Bay Buccaneers,QB,20922000
Marcus Mariota,Tennessee Titans,QB,20922000
Derek Carr,Oakland Raiders,QB,19900000
Jimmy Garoppolo,San Francisco 49Ers,QB,17200000


In [19]:
# Querying a non-unique index: all players on the 'New York Jets'
nfl_by_team = nfl.reset_index().set_index("Team")
jets_players = nfl_by_team.loc["New York Jets"]
print("Total Jets players in dataset:", len(jets_players))
display(jets_players.head(3))

Total Jets players in dataset: 58


,Name,Position,Birthday,Salary
Team,,,,
New York Jets,Bronson Kaufusi,DE,1991-07-06,645000
New York Jets,Darryl Roberts,CB,1990-11-26,1000000
New York Jets,Jordan Willis,DE,1995-05-02,754750


## 10. DataFrame Architecture Cheat Sheet

| Operation | Syntax | Key Advantage |
| :--- | :--- | :--- |
| **Row by Position** | `df.iloc[i]` | 0-based offset |
| **Row by Label** | `df.loc[label]` | Returns Series (if unique) or DataFrame (if duplicate) |
| **Scalar Cell** | `df.at[label, col]` | Up to 20x faster than `.loc` |
| **Scalar Positional** | `df.iat[i, j]` | Up to 20x faster than `.iloc` |
| **Safe Renaming** | `df.rename(columns={...})` | Targeted, immune to column count mismatches |
| **Memory Saver** | `df['col'].astype('category')` | Ideal for low-cardinality categories (<1% unique) |
| **Numeric Reductions**| `df.mean(numeric_only=True)` | Prevents TypeError on string/datetime columns |

---
## 🎯 11. Technical Interview Corner: Tricky Questions & Drills

### Q1: The Duplicate Index Return Type Dilemma
**Question**: What is the return type of `df.loc['X']`?

**Answer**:
- If `'X'` is **unique** in the index: returns a **1D `pd.Series`**.
- If `'X'` appears **multiple times**: returns a **2D `pd.DataFrame`**!
- If `'X'` does **not exist**: raises a `KeyError`.
*Interview Insight*: In production code, assuming `df.loc['X']` always returns a Series can lead to subtle bugs when dirty data contains duplicate keys.

In [20]:
# Demonstration of duplicate index return type
demo_unique = pd.DataFrame({"Val": [10, 20]}, index=["A", "B"])
demo_dup = pd.DataFrame({"Val": [10, 20, 30]}, index=["A", "A", "B"])

print("Unique index lookup type:    ", type(demo_unique.loc["A"]))
print("Duplicate index lookup type: ", type(demo_dup.loc["A"]))

Unique index lookup type:     <class 'pandas.Series'>
Duplicate index lookup type:  <class 'pandas.DataFrame'>


### Q2: Why does `df.columns = ['a', 'b', 'c']` introduce silent failure risks?
**Question**: Explain why directly mutating `df.columns` with a list is considered risky in production data engineering pipelines.

**Answer**:
1. **Silent Positional Misalignment**: If an upstream ETL change adds a column or reorders columns, assigning a list does not fail—it silently assigns wrong column headers to completely unrelated data!
2. **Lack of Immutability / Chaining**: Direct assignment cannot be chained in method pipelines (`df.rename(...).query(...)`).

### Q3: Highest Paid Player per Position
**Challenge**: In a single chained expression, find the highest-paid player for each distinct `Position` in the NBA dataset, sorted by `Salary` descending.

In [21]:
# Solution using sort_values and drop_duplicates
top_earner_by_pos = (
    nba.dropna(subset=["Salary"])
    .sort_values(by="Salary", ascending=False)
    .drop_duplicates(subset=["Position"])
    [["Position", "Name", "Team", "Salary"]]
    .sort_values(by="Salary", ascending=False)
    .reset_index(drop=True)
)

top_earner_by_pos

,Position,Name,Team,Salary
0,PG,Stephen Curry,Golden State Warriors,40231758
1,PF,LeBron James,Los Angeles Lakers,37436858
2,SF,Paul George,Los Angeles Clippers,33005556
3,SG,Klay Thompson,Golden State Warriors,32742000
4,C,Kevin Love,Cleveland Cavaliers,28942830
5,F,Zion Williamson,New Orleans Pelicans,9757440
6,G,Ty Jerome,Phoenix Suns,2193480
7,GF,Dylan Windler,Cleveland Cavaliers,2035800
8,FC,Norvel Pelle,Philadelphia 76ers,79568
